# Behavioral Drift Detection for Online Exams

This notebook demonstrates how to use the behavioral drift detection pipeline.

## 1. Setup Environment

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import load_config, set_seed, get_device
from src.data_loader import EdNetDataLoader, SessionCreator, OULADLoader
from src.feature_extraction import BehavioralFeatureExtractor
from src.models import LSTMAutoencoder

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Load Configuration

In [ ]:
# Load config
config = load_config('../configs/config.yaml')
set_seed(config['training']['seed'])
device = get_device(config['training']['device'])

print("Configuration loaded successfully!")
print(f"Device: {device}")

## 3. Load Sample Data

In [ ]:
# Load EdNet data (first 100 students for quick demo)
ednet_loader = EdNetDataLoader(config['data']['ednet_path'])
students_data = ednet_loader.load_batch(start_idx=0, batch_size=100)

print(f"Loaded {len(students_data)} students")

## 4. Create Exam Sessions

In [ ]:
# Create sessions
session_creator = SessionCreator(
    min_questions=config['data']['session_min_questions'],
    max_questions=config['data']['session_max_questions']
)

sessions, student_ids = session_creator.create_all_sessions(students_data)

print(f"Created {len(sessions)} sessions")
print(f"\nSample session shape: {sessions[0].shape}")
print(f"Session columns: {list(sessions[0].columns)}")

## 5. Extract Behavioral Features

In [ ]:
# Extract features
extractor = BehavioralFeatureExtractor()
feature_matrix, feature_dicts = extractor.extract_features_batch(sessions)

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Features: {extractor.feature_names}")

## 6. Visualize Features

In [ ]:
# Create feature DataFrame
feature_df = pd.DataFrame(feature_matrix, columns=extractor.feature_names)

# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(extractor.feature_names):
    axes[i].hist(feature_df[feature].dropna(), bins=30, edgecolor='black')
    axes[i].set_title(f'{feature}')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 7. Feature Correlation

In [ ]:
# Compute correlation matrix
corr_matrix = feature_df.corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 8. Load Pretrained Model (if available)

In [ ]:
# Initialize LSTM Autoencoder
model = LSTMAutoencoder(
    input_dim=config['model']['input_dim'],
    hidden_dim=config['model']['hidden_dim'],
    latent_dim=config['model']['latent_dim'],
    num_layers=config['model']['num_layers'],
    dropout=config['model']['dropout']
).to(device)

print(f"Model initialized with {sum(p.numel() for p in model.parameters())} parameters")

# Try to load pretrained weights
try:
    model.load_state_dict(torch.load('../results/models/best_model.pth', map_location=device))
    print("\nLoaded pretrained model!")
except:
    print("\nNo pretrained model found. Run main.py to train.")

## 9. Make Predictions

In [ ]:
# Normalize features
feature_matrix_norm, feat_mean, feat_std = extractor.normalize_features(feature_matrix)

# Prepare for LSTM (add sequence dimension)
feature_tensor = torch.FloatTensor(feature_matrix_norm).unsqueeze(1).to(device)

# Get reconstructions
model.eval()
with torch.no_grad():
    reconstructed = model(feature_tensor)
    reconstruction_errors = torch.mean((feature_tensor - reconstructed) ** 2, dim=(1, 2))

print(f"Reconstruction errors computed for {len(reconstruction_errors)} sessions")
print(f"Mean error: {reconstruction_errors.mean():.4f}")
print(f"Std error: {reconstruction_errors.std():.4f}")

## 10. Visualize Reconstruction Errors

In [ ]:
errors_np = reconstruction_errors.cpu().numpy()

# Plot distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(errors_np, bins=50, edgecolor='black')
plt.axvline(errors_np.mean(), color='r', linestyle='--', label='Mean')
plt.axvline(errors_np.mean() + 2*errors_np.std(), color='orange', 
           linestyle='--', label='Mean + 2σ')
plt.xlabel('Reconstruction Error')
plt.ylabel('Frequency')
plt.title('Distribution of Reconstruction Errors')
plt.legend()

plt.subplot(1, 2, 2)
plt.boxplot(errors_np, vert=True)
plt.ylabel('Reconstruction Error')
plt.title('Reconstruction Error Box Plot')

plt.tight_layout()
plt.show()

## 11. Identify Top Anomalies

In [ ]:
# Get top 10 most anomalous sessions
top_k = 10
top_indices = np.argsort(errors_np)[-top_k:]

print("Top 10 Most Anomalous Sessions:")
print("="*60)
for rank, idx in enumerate(reversed(top_indices), 1):
    print(f"\nRank {rank}: Session {idx}")
    print(f"  Reconstruction Error: {errors_np[idx]:.4f}")
    print(f"  Student ID: {student_ids[idx]}")
    print(f"  Features:")
    for feat_name, feat_val in zip(extractor.feature_names, feature_matrix[idx]):
        print(f"    {feat_name}: {feat_val:.4f}")

## Next Steps

To run the full pipeline:

```bash
# Full pipeline (preprocess + train + evaluate + fairness + explain)
python main.py --config configs/config.yaml --mode all

# Or run individual steps:
python main.py --mode preprocess
python main.py --mode train
python main.py --mode evaluate
python main.py --mode fairness
python main.py --mode explain
```